# Step 1.2b – NLP Features for DoubleML
**Thesis: Geopolitical Turning Points and Macroeconomic Volatility – Extension of Saadaoui (2026)**

---

## What this notebook does

Builds the **high‑dimensional NLP controls** required by the Saadaoui ML extension plan.
Uses **Option A (structured substitution)**:

| Plan requirement | Our variable | Justification |
|---|---|---|
| NLP sentiment (FinBERT) | `gdelt_goldstein_mean` | Goldstein scores are expert‑coded valence (−10 to +10) directly from the same news articles FinBERT would read. |
| Event counts | `protest_share`, `military_share`, `diplomatic_share` | Normalised monthly shares to avoid news‑volume bias. |
| Topic model proportions | `gdelt_topic_01` … `gdelt_topic_20` | CAMEO root code shares – a 20‑category probability distribution structurally identical to LDA output. |
| Additional | `gdelt_sentiment_signal`, `gdelt_conflict_share`, `gdelt_coop_share` | Interpretable aggregate features.

### Covered data sources
- **GDELT** – already extracted and verified (`gdel_events_monthly_clean.csv`)
- **GPR** – Caldara‑Iacoviello geopolitical risk index
- **EA‑GPR** – Bondarenko euro‑area geopolitical risk
- **WUI** – World Uncertainty Index (local quarterly CSV → monthly)

### Output
- `data/03_nlp/feature_matrix_nlp_A.csv` – full merged feature matrix
- `data/03_nlp/var_roles_nlp_A.json` – updated variable roles
- `data/03_nlp/coverage_report_A.csv` – coverage audit

**No external downloads are required beyond what you already have.**


---
## Setup

In [1]:
import pandas as pd
import numpy as np
import requests
import json
import io
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / 'data').exists() else NOTEBOOK_DIR
DATA_DIR     = PROJECT_ROOT / 'data'
NLP_DIR      = DATA_DIR / '03_nlp'
RAW_DIR      = NLP_DIR  / 'raw'
PROC_DIR     = DATA_DIR / '02_features'

NLP_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── Input files ───────────────────────────────────────────────────────────────
GDELT_CSV    = NLP_DIR  / 'gdel_events_monthly_clean.csv'
FEAT_MATRIX  = PROC_DIR / 'feature_matrix.csv'
VAR_ROLES_IN = PROC_DIR / 'var_roles.json'

# ── Output files ─────────────────────────────────────────────────────────────
GPR_CSV      = NLP_DIR / 'gpr_monthly.csv'
EA_GPR_CSV   = NLP_DIR / 'ea_gpr_monthly.csv'
WUI_CSV      = NLP_DIR / 'wui_monthly.csv'   # already created from quarterly
OUT_FEAT     = NLP_DIR / 'feature_matrix_nlp_A.csv'
OUT_ROLES    = NLP_DIR / 'var_roles_nlp_A.json'
COV_REPORT   = NLP_DIR / 'coverage_report_A.csv'

# ── Sample window ─────────────────────────────────────────────────────────────
START      = '1990-01-01'
END        = '2022-02-01'
date_range = pd.date_range(START, END, freq='MS')

def align(df):
    df = df.copy()
    df.index = pd.to_datetime(df.index).to_period('M').to_timestamp()
    return df.reindex(date_range)

# Verify inputs
for path, label in [(GDELT_CSV,'GDELT CSV'), (FEAT_MATRIX,'Feature matrix'), (VAR_ROLES_IN,'var_roles.json')]:
    status = 'FOUND' if path.exists() else 'MISSING'
    print(f'  [{status}] {label}: {path}')
print(f'\nSample window: {START} to {END} ({len(date_range)} months)')


  [FOUND] GDELT CSV: C:\Users\HP\Desktop\replication+contribution\data\03_nlp\gdel_events_monthly_clean.csv
  [FOUND] Feature matrix: C:\Users\HP\Desktop\replication+contribution\data\02_features\feature_matrix.csv
  [FOUND] var_roles.json: C:\Users\HP\Desktop\replication+contribution\data\02_features\var_roles.json

Sample window: 1990-01-01 to 2022-02-01 (386 months)


---
## Section 1 – Process GDELT features into ML‑ready proportions

In [2]:
# ── Load GDELT CSV ────────────────────────────────────────────────────────────
df_gdelt_raw = pd.read_csv(GDELT_CSV, index_col=0, parse_dates=True)
df_gdelt_raw = align(df_gdelt_raw)
print(f'GDELT loaded: {df_gdelt_raw.shape[0]} months x {df_gdelt_raw.shape[1]} columns')
print(f'Coverage: {df_gdelt_raw["total_events"].notna().sum()}/386 months')
print()

# ── Build feature DataFrame ────────────────────────────────────────────────────
df_nlp = pd.DataFrame(index=date_range)

# 1. Total events (log scale to reduce right‑skew caused by growing news volume)
df_nlp['gdelt_total_events_log']  = np.log1p(df_gdelt_raw['total_events'])

# 2. Event‑type shares
df_nlp['gdelt_protest_share']     = df_gdelt_raw['protest_count']    / df_gdelt_raw['total_events']
df_nlp['gdelt_military_share']    = df_gdelt_raw['military_count']   / df_gdelt_raw['total_events']
df_nlp['gdelt_diplomatic_share']  = df_gdelt_raw['diplomatic_count'] / df_gdelt_raw['total_events']

# 3. Sentiment proxy: Goldstein mean (expert‑coded, −10 conflict to +10 cooperation)
df_nlp['gdelt_goldstein_mean']    = df_gdelt_raw['goldstein_mean']

# 4. Sentiment signal (abs tone × log total events) – captures certainty of sentiment
df_nlp['gdelt_sentiment_signal']  = (
    df_gdelt_raw['goldstein_mean'].abs() * np.log1p(df_gdelt_raw['total_events'])
)

# 5. CAMEO root code shares (topic_01 … topic_20)
cameo_labels = {
    1:'statement', 2:'appeal', 3:'intent_cooperate', 4:'consult',
    5:'diplomatic_coop', 6:'material_coop', 7:'provide_aid', 8:'yield',
    9:'investigate', 10:'demand', 11:'disapprove', 12:'reject',
    13:'threaten', 14:'protest', 15:'exhibit_force', 16:'reduce_relations',
    17:'coerce', 18:'assault', 19:'fight', 20:'mass_violence'
}

for i in range(1, 21):
    col_new = f'gdelt_topic_{i:02d}_{cameo_labels[i]}'
    col_raw = f'theme_{i}'
    df_nlp[col_new] = df_gdelt_raw[col_raw] / df_gdelt_raw['total_events']

# 6. Aggregate conflict / cooperation shares
df_nlp['gdelt_conflict_share'] = (
    df_gdelt_raw[['theme_18','theme_19','theme_20']].sum(axis=1)
    / df_gdelt_raw['total_events']
)
df_nlp['gdelt_coop_share'] = (
    df_gdelt_raw[['theme_3','theme_4','theme_5','theme_6','theme_7','theme_8']].sum(axis=1)
    / df_gdelt_raw['total_events']
)
df_nlp['gdelt_hostility_share'] = (
    df_gdelt_raw[['theme_13','theme_14','theme_15','theme_16','theme_17','theme_18','theme_19','theme_20']].sum(axis=1)
    / df_gdelt_raw['total_events']
)

print('GDELT features built:')
for col in df_nlp.columns:
    n   = df_nlp[col].notna().sum()
    pct = 100 * n / 386
    rng = f'[{df_nlp[col].min():.3f}, {df_nlp[col].max():.3f}]'
    print(f'  {col:45s}: {n}/386 ({pct:.0f}%)  range {rng}')


GDELT loaded: 386 months x 26 columns
Coverage: 386/386 months

GDELT features built:
  gdelt_total_events_log                       : 386/386 (100%)  range [3.135, 10.047]
  gdelt_protest_share                          : 386/386 (100%)  range [0.000, 0.099]
  gdelt_military_share                         : 386/386 (100%)  range [0.000, 0.091]
  gdelt_diplomatic_share                       : 386/386 (100%)  range [0.267, 0.824]
  gdelt_goldstein_mean                         : 386/386 (100%)  range [-0.733, 4.789]
  gdelt_sentiment_signal                       : 386/386 (100%)  range [0.107, 27.899]
  gdelt_topic_01_statement                     : 386/386 (100%)  range [0.014, 0.210]
  gdelt_topic_02_appeal                        : 386/386 (100%)  range [0.000, 0.186]
  gdelt_topic_03_intent_cooperate              : 386/386 (100%)  range [0.000, 0.261]
  gdelt_topic_04_consult                       : 386/386 (100%)  range [0.096, 0.636]
  gdelt_topic_05_diplomatic_coop               : 38

---
## Section 2 – Load GPR, EA‑GPR, WUI (already present)

In [3]:
# ── GPR Index ────────────────────────────────────────────────────────────────
if GPR_CSV.exists():
    df_gpr = pd.read_csv(GPR_CSV, index_col=0, parse_dates=True)
    df_gpr = align(df_gpr)
    print(f'GPR: loaded from cache. Coverage: {df_gpr["gpr"].notna().sum()}/386')
else:
    raise FileNotFoundError(f'Run 03_nlp_features_v_please.ipynb first to create {GPR_CSV}')

# Sanity check: mean around 90, std around 67
if df_gpr['gpr'].notna().any():
    print(f'  GPR mean={df_gpr["gpr"].mean():.1f}  std={df_gpr["gpr"].std():.1f}')


GPR: loaded from cache. Coverage: 381/386
  GPR mean=89.3  std=66.6


In [4]:
# ── EA‑GPR Index ─────────────────────────────────────────────────────────────
if EA_GPR_CSV.exists():
    df_ea = pd.read_csv(EA_GPR_CSV, index_col=0, parse_dates=True)
    df_ea = align(df_ea)
    print(f'EA‑GPR: loaded from cache. Coverage: {df_ea["ea_gpr"].notna().sum()}/386 (starts 1998)')
else:
    raise FileNotFoundError(f'Run 03_nlp_features_v_please.ipynb first to create {EA_GPR_CSV}')


EA‑GPR: loaded from cache. Coverage: 290/386 (starts 1998)


In [5]:
# ── World Uncertainty Index (local quarterly → monthly) ──────────────────────
if WUI_CSV.exists():
    df_wui = pd.read_csv(WUI_CSV, index_col=0, parse_dates=True)
    df_wui = align(df_wui)
    print(f'WUI: loaded from cache. Coverage: {df_wui["wui"].notna().sum()}/386')
    print(f'  WUI mean={df_wui["wui"].mean():.1f}  range={df_wui["wui"].min():.0f}–{df_wui["wui"].max():.0f}')
else:
    raise FileNotFoundError(f'Run 03_nlp_features_v_please.ipynb first to create {WUI_CSV}')


WUI: loaded from cache. Coverage: 386/386
  WUI mean=17400.8  range=5570–55685


---
## Section 3 – Coverage Report

In [6]:
print('=' * 70)
print('COVERAGE REPORT – NLP Features')
print('=' * 70)

all_dfs = [
    (df_nlp, 'GDELT structured'),
    (df_gpr, 'GPR (Caldara‑Iacoviello)'),
    (df_ea,  'EA‑GPR (Bondarenko)'),
    (df_wui, 'WUI (World Uncertainty)'),
]

report_rows = []
for df, label in all_dfs:
    for col in df.columns:
        n     = df[col].notna().sum()
        pct   = 100 * n / 386
        start = str(df[col].first_valid_index())[:7] if df[col].notna().any() else 'N/A'
        end   = str(df[col].last_valid_index())[:7]  if df[col].notna().any() else 'N/A'
        rec   = 'PRIMARY' if pct >= 80 else ('ROBUSTNESS' if pct >= 20 else 'DROP')
        report_rows.append({'source':label,'variable':col,'n_months':n,
                            'pct_filled':round(pct,1),'start':start,'end':end,'recommend':rec})

df_report = pd.DataFrame(report_rows)
print(df_report.to_string(index=False))
df_report.to_csv(COV_REPORT, index=False)
print(f'\nSaved: {COV_REPORT}')

# Sanity checks
print('\nSANITY CHECKS')
print('-' * 50)
checks = [
    ('GDELT 100% coverage',       df_nlp['gdelt_total_events_log'].notna().sum() == 386),
    ('Topic shares sum ~1',       abs(df_nlp[[c for c in df_nlp.columns if 'topic_' in c]].sum(axis=1).mean() - 1.0) < 0.05),
    ('Goldstein in range -10..10',df_nlp['gdelt_goldstein_mean'].between(-10,10).all()),
    ('GPR coverage > 99%',        df_gpr['gpr'].notna().sum() >= 381),
    ('GPR plausible (mean 50-200)', 50 <= df_gpr['gpr'].mean() <= 200),
    ('WUI plausible (mean < 1)',   df_wui['wui'].mean() < 1.0),
]
all_ok = True
for label, result in checks:
    icon = 'PASS' if result else 'FAIL'
    print(f'  [{icon}] {label}')
    if not result: all_ok = False
print('\nAll sanity checks passed.' if all_ok else '\nSome checks failed – review before merging.')


COVERAGE REPORT – NLP Features
                  source                        variable  n_months  pct_filled   start     end  recommend
        GDELT structured          gdelt_total_events_log       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured             gdelt_protest_share       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured            gdelt_military_share       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured          gdelt_diplomatic_share       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured            gdelt_goldstein_mean       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured          gdelt_sentiment_signal       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured        gdelt_topic_01_statement       386       100.0 1990-01 2022-02    PRIMARY
        GDELT structured           gdelt_topic_02_appeal       386       100.0 1990-01 2022-02    PRIMARY
        GDELT s

---
## Section 4 – Merge with macro feature matrix and save

In [7]:
# ── Load macro feature matrix ────────────────────────────────────────────────
if not FEAT_MATRIX.exists():
    raise FileNotFoundError(f'Run 02_feature_matrix.ipynb first. Not found: {FEAT_MATRIX}')

df_base = pd.read_csv(FEAT_MATRIX, index_col=0, parse_dates=True)
df_base.index = pd.to_datetime(df_base.index).to_period('M').to_timestamp()
print(f'Macro feature matrix: {df_base.shape[0]} obs x {df_base.shape[1]} cols')

# ── Merge: lag ALL NLP features by 1 month (look‑ahead prevention) ───────────
df_merged = df_base.copy()
for df_src, label in [
    (df_nlp, 'GDELT features'),
    (df_gpr, 'GPR'),
    (df_ea,  'EA‑GPR'),
    (df_wui, 'WUI'),
]:
    df_lagged = df_src.shift(1)
    for col in df_lagged.columns:
        df_merged[col] = df_lagged[col]
    print(f'  Merged {label}: {len(df_src.columns)} columns')

print(f'\nMerged shape: {df_merged.shape[0]} obs x {df_merged.shape[1]} cols')
assert df_merged.shape[0] == 386, 'Row count wrong'
assert 'lbrent' not in df_merged.columns, 'lbrent leaked into final matrix'

# Save
df_merged.to_csv(OUT_FEAT)
print(f'Saved: {OUT_FEAT}')


Macro feature matrix: 386 obs x 68 cols
  Merged GDELT features: 29 columns
  Merged GPR: 3 columns
  Merged EA‑GPR: 2 columns
  Merged WUI: 1 columns

Merged shape: 386 obs x 103 cols
Saved: C:\Users\HP\Desktop\replication+contribution\data\03_nlp\feature_matrix_nlp_A.csv


In [8]:
# ── Update var_roles.json with NLP control lists ──────────────────────────────
with open(VAR_ROLES_IN) as f:
    var_roles = json.load(f)

topic_cols = [c for c in df_nlp.columns if 'topic_' in c]

var_roles['controls_gdelt_event']    = ['gdelt_total_events_log','gdelt_protest_share',
                                         'gdelt_military_share','gdelt_diplomatic_share',
                                         'gdelt_conflict_share','gdelt_coop_share',
                                         'gdelt_hostility_share']
var_roles['controls_gdelt_sentiment']= ['gdelt_goldstein_mean','gdelt_sentiment_signal']
var_roles['controls_gdelt_topics']   = topic_cols
var_roles['controls_gpr']            = ['gpr','gpr_threats','gpr_acts']
var_roles['controls_ea_gpr']         = ['ea_gpr','ea_gpr_diff']
var_roles['controls_wui']            = ['wui']

# PRIMARY NLP spec: full‑coverage features
var_roles['controls_nlp_primary'] = (
    var_roles['controls_gdelt_event']
    + var_roles['controls_gdelt_sentiment']
    + var_roles['controls_gdelt_topics']
    + var_roles['controls_gpr']
    + var_roles['controls_wui']
)

# ROBUSTNESS NLP spec: adds EA‑GPR (75% coverage)
var_roles['controls_nlp_robustness'] = (
    var_roles['controls_nlp_primary']
    + var_roles['controls_ea_gpr']
)

# Complete ML control lists
var_roles['controls_all_nlp_primary'] = (
    var_roles.get('controls_all_ml_dense', [])
    + var_roles['controls_nlp_primary']
)
var_roles['controls_all_nlp_robustness'] = (
    var_roles.get('controls_all_ml_dense', [])
    + var_roles['controls_nlp_robustness']
)

with open(OUT_ROLES, 'w') as f:
    json.dump(var_roles, f, indent=2)
print(f'Saved: {OUT_ROLES}')
print()
print('Control list summary:')
for k in ['controls_nlp_primary','controls_nlp_robustness',
          'controls_all_nlp_primary','controls_all_nlp_robustness']:
    print(f'  {k:40s}: {len(var_roles[k])} variables')


Saved: C:\Users\HP\Desktop\replication+contribution\data\03_nlp\var_roles_nlp_A.json

Control list summary:
  controls_nlp_primary                    : 33 variables
  controls_nlp_robustness                 : 35 variables
  controls_all_nlp_primary                : 62 variables
  controls_all_nlp_robustness             : 64 variables


---
## Done

**Output files:**
- `feature_matrix_nlp_A.csv` – the merged dataset (386 obs) ready for DoubleML
- `var_roles_nlp_A.json` – variable roles with NLP controls included
- `coverage_report_A.csv` – coverage audit

**Next step:** load these into the DoubleML notebook and estimate impulse responses.